# 我的外送還要多久才會到？
**從生活認識 AI｜長榮大學資訊工程學系｜2026.12.05**

外送 App 顯示「預估 25 分鐘送達」，這個數字是怎麼來的？

這次，我們先把問題簡化：**只用外送距離，預測送達時間。**
你會親手完成一次機器學習：

**看資料 → 畫圖 → 訓練 → 檢查誤差 → 預測新訂單。**

本次使用的是整理好的模擬外送訂單，共 1,000 筆。
先依序執行、觀察結果，不需要一開始就背熟每一行程式。

從第 1 步開始，依序按下每個程式區塊左側的 **▶**。
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andy6804tw/speech-archive/blob/main/2026/20261205-%E9%95%B7%E6%A6%AE%E5%A4%A7%E5%AD%B8%E8%B3%87%E5%B7%A5%E7%B3%BB-AI%E5%AF%A6%E4%BD%9C%E8%88%87%E5%AD%B8%E9%95%B7%E7%B6%93%E9%A9%97%E5%88%86%E4%BA%AB/notebooks/%E5%A4%96%E9%80%81%E6%99%82%E9%96%93%E9%A0%90%E6%B8%AC/%E5%A4%96%E9%80%81%E9%82%84%E8%A6%81%E5%A4%9A%E4%B9%85%E6%89%8D%E6%9C%83%E5%88%B0.ipynb)


## 1. 看看過去的訂單

每一列是一張過去的訂單：

- **距離_公里**：這張訂單要送多遠。
- **送達時間_分鐘**：這張訂單最後實際花了多久。

下方程式使用 `pd.read_csv()`，直接從 GitHub 讀取課程整理好的 CSV。

執行後，先看前 5 張訂單。距離與時間，哪一個是我們想讓模型預測的答案？


In [ ]:
import pandas as pd

data_url = 'https://raw.githubusercontent.com/andy6804tw/speech-archive/main/2026/20261205-%E9%95%B7%E6%A6%AE%E5%A4%A7%E5%AD%B8%E8%B3%87%E5%B7%A5%E7%B3%BB-AI%E5%AF%A6%E4%BD%9C%E8%88%87%E5%AD%B8%E9%95%B7%E7%B6%93%E9%A9%97%E5%88%86%E4%BA%AB/notebooks/%E5%A4%96%E9%80%81%E6%99%82%E9%96%93%E9%A0%90%E6%B8%AC/%E8%B3%87%E6%96%99%E9%9B%86/delivery_distance_time.csv'
data = pd.read_csv(data_url)
data.head()


## 2. 把訂單畫成圖

這一步使用 Matplotlib 畫圖。圖上的每一個點，都是一張訂單。
**橫軸是距離（公里），縱軸是實際送達時間（分鐘）。**

觀察看看：距離較遠的訂單，通常會花比較久嗎？同樣的距離，每一張訂單都一樣快嗎？

先執行下面的「繪圖設定」，讓圖表顯示繁體中文，再執行畫圖的程式。

In [ ]:
# @title 繪圖設定（點左側 ▶）
%matplotlib inline
from pathlib import Path
from urllib.request import urlopen
from matplotlib import font_manager, rcParams

font_path = Path('字型/NotoSansCJKtc-Regular.otf')
if not font_path.exists():
    font_path.parent.mkdir(parents=True, exist_ok=True)
    font_url = 'https://raw.githubusercontent.com/notofonts/noto-cjk/165c01b46ea533872e002e0785ff17e44f6d97d8/Sans/OTF/TraditionalChinese/NotoSansCJKtc-Regular.otf'
    with urlopen(font_url, timeout=60) as response:
        font_bytes = response.read()
    font_path.write_bytes(font_bytes)

font_manager.fontManager.addfont(str(font_path))
font_name = font_manager.FontProperties(fname=str(font_path)).get_name()
rcParams.update({'font.family': font_name, 'font.size': 12, 'axes.unicode_minus': False})

def plot_orders(data, model=None):
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.scatter(data['距離_公里'], data['送達時間_分鐘'],
               s=16, alpha=0.35, color='#59758c', label='每個點代表一張訂單')
    if model is not None:
        line_x = pd.DataFrame({'距離_公里': [data['距離_公里'].min(), data['距離_公里'].max()]})
        ax.plot(line_x['距離_公里'], model.predict(line_x),
                color='#df9b00', linewidth=3, label='模型學到的規律')
    ax.set(title='外送距離與送達時間', xlabel='外送距離（公里）', ylabel='送達時間（分鐘）')
    if model is not None:
        formula = f'預估時間 ≈ {model.coef_[0]:.4f} × 距離 + {model.intercept_:.4f}'
        ax.set_title('外送距離與送達時間\n' + formula)
    ax.spines[['top', 'right']].set_visible(False)
    ax.legend(frameon=False)
    fig.tight_layout()
    plt.show()

print('繪圖設定完成！接著執行下一格，把訂單畫成圖。')

In [ ]:
import matplotlib.pyplot as plt

plot_orders(data)

## 3. 讓模型從訂單中學習

把過去訂單的「距離」與「實際送達時間」交給模型，讓它學習兩者的關係。

先用 **800 筆訂單練習**，另外 **200 筆留到下一步檢查**。

執行下面的程式，看看模型會學出什麼。

In [ ]:
from sklearn.model_selection import train_test_split

X = data[['距離_公里']]
y = data['送達時間_分鐘']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

from sklearn.linear_model import LinearRegression

model = LinearRegression()  # 建立線性迴歸模型
model.fit(X_train, y_train)

a = model.coef_[0]   # 取出斜率
b = model.intercept_  # 取出截距

print(f'模型學到的參數：a = {a:.4f}，b = {b:.4f}')
print(f'預估時間（分鐘）≈ {a:.4f} × 距離（公里）+ {b:.4f}')
plot_orders(X_train.assign(送達時間_分鐘=y_train), model)

### 模型學到了什麼？

看看剛才的黃色直線，它可以寫成熟悉的直線公式：

$$y = ax + b$$

**x 是外送距離，y 是預估送達時間**。對照上面印出的兩個參數：

- **a：係數，也就是斜率**。決定直線的方向與傾斜程度，也表示距離每增加 1 公里，模型預估的時間增加幾分鐘。
- **b：截距**。調整整條直線的上下位置；固定 a 時，b 越大，線越往上移。

我們選用的 `LinearRegression`（線性迴歸），就是透過 `fit()` 從資料學出 a 與 b。
`model.coef_[0]` 可以取出 a，`model.intercept_` 可以取出 b。

**模型訓練，就是從訂單資料學出：這條線要多斜（a）、放多高（b）。**

## 4. 遇到沒看過的訂單，平均會差幾分鐘？

現在請模型預測剛才留下的 200 筆訂單，再與實際時間比較。
程式中的 `predict()`，就是請模型做出預測。

例如實際等了 25 分鐘，模型猜 24 分鐘，這張訂單就相差 1 分鐘。
把每張訂單相差的分鐘數取平均，就能知道模型整體大約差多少。

先看看前 5 筆的預估與實際時間，再讀下面的平均誤差。

In [ ]:
from sklearn.metrics import mean_absolute_error

predictions = model.predict(X_test)
comparison = pd.DataFrame({
    '實際時間（分鐘）': y_test.to_numpy(),
    '模型預估（分鐘）': predictions
})
display(comparison.head().round(1))

error = mean_absolute_error(y_test, predictions)
print(f'在這 200 筆訂單中，預估與實際時間平均相差約 {error:.1f} 分鐘。')

## 5. 換你下一張訂單！

假設你住在店家 **3 公里**外，模型會猜多久？

用剛才學到的 a 與 b，自己代入 **預估時間 = a × 距離 + b** 算一次，
再與 `model.predict()` 的結果比較。兩種方式使用同一組參數，會得到相同的預估。

用滑桿調整下一格的距離，再按 **▶**。也可以直接修改程式中的數字。
試試 **2、5、8 公里**：先猜時間會怎麼變，再執行看看。
本次先使用資料範圍內的距離：0.6～19.9 公里。

In [ ]:
# @title 輸入外送距離（公里），再按 ▶
distance_km = 3.0 # @param {type:"slider", min:0.6, max:19.9, step:0.1}

if 0.6 <= distance_km <= 19.9:
    formula_time = a * distance_km + b  # 自己代入公式計算

    new_order = pd.DataFrame({'距離_公里': [distance_km]})
    predicted_time = model.predict(new_order)[0]

    print(f'距離店家 {distance_km:g} 公里：')
    print(f'公式計算：{a:.4f} × {distance_km:g} + {b:.4f} ≈ {formula_time:.1f} 分鐘')
    print(f'model.predict()：模型預估約 {predicted_time:.1f} 分鐘送達。')
    print('這是模型的預估，實際送達時間仍可能不同。')
else:
    print('請輸入 0.6～19.9 公里之間的距離，再執行一次。')

## 完成！你已經做出一個外送時間預測模型

你剛才做了三件事：

1. 用過去的訂單，讓模型學出直線公式中的 a 與 b。
2. 用沒看過的訂單，檢查預測會差多少。
3. 把新的距離代入公式，並確認與 `predict()` 的結果相同。

**想一想：如果距離一樣，但一張訂單遇到下雨、一張沒有，時間會一樣嗎？**
如果希望模型能考慮這些差異，你還想提供哪些資訊？

資料來源：[Food Delivery Time Prediction](https://www.kaggle.com/datasets/denkuznetz/food-delivery-time-prediction)。
本教材使用其中的模擬資料，整理成距離與送達時間兩欄。